# IMPLEMENTACIÓN Y SIMULACIÓN DEL PERCEPTRÓN SIMPLE DESDE CERO

**Perceptrón de Rosenblatt implementado desde cero con `numpy` y `matplotlib`**

- **Materia:** `[Nombre de la materia]`
- **Equipo:** `[Integrantes del equipo]`
- **Fecha:** `[Fecha de entrega]`

En este cuaderno se implementa desde cero una neurona artificial de un solo nivel (el **Perceptrón de Rosenblatt**, 1958) usando **únicamente** la librería `numpy` para la lógica matemática, el entrenamiento y la predicción, y **únicamente** `matplotlib` para la visualización. Se simula su comportamiento en tres problemas de clasificación binaria: las compuertas lógicas **AND**, **OR** y **XOR**.

## Objetivos de aprendizaje

1. **Comprender** la arquitectura matemática interna de una neurona artificial (Perceptrón de Rosenblatt).
2. **Implementar** el algoritmo de entrenamiento del Perceptrón utilizando únicamente álgebra vectorial básica (`numpy`).
3. **Simular y evaluar** el comportamiento del Perceptrón en problemas de separación lineal: compuertas lógicas **AND**, **OR** y el problema **XOR** (no linealmente separable).

In [ ]:
# ============================================================================
# IMPORTACIÓN DE LIBRERÍAS
# - numpy      -> lógica matemática, entrenamiento y predicción (único framework permitido)
# - matplotlib -> visualización de resultados
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt

# Configuración estética global de las figuras
plt.rcParams["figure.dpi"] = 110

print("Librerías importadas correctamente.")
print("Versión de NumPy:", np.__version__)

## Fundamento teórico resumido

El Perceptrón recibe un vector de entrada $\mathbf{x} = [x_1, x_2, \dots, x_n]^T$, calcula una **combinación lineal ponderada** con el vector de pesos $\mathbf{w} = [w_1, w_2, \dots, w_n]^T$ más un **sesgo (bias)** $b$, y pasa el resultado a través de una **función escalón de paso (Heaviside)**:

$$
z = \sum_{i=1}^{n} w_i \, x_i + b = \mathbf{w}^T \mathbf{x} + b
$$

$$
\hat{y} = f(z) =
\begin{cases}
1 & \text{si } z \geq 0 \\[2mm]
0 & \text{si } z < 0
\end{cases}
$$

### Regla de actualización de pesos (aprendizaje)

Para cada muestra de entrenamiento, el error se calcula como $e = y - \hat{y}$. Si existe error ($e \neq 0$), los pesos y el sesgo se actualizan según la regla de Rosenblatt:

$$
w_j \leftarrow w_j + \eta \cdot (y - \hat{y}) \cdot x_j
$$

$$
b \leftarrow b + \eta \cdot (y - \hat{y})
$$

Donde $\eta$ es la **tasa de aprendizaje** (*learning rate*), con $0 < \eta \le 1$.

# PARTE 1: Construcción de la Clase `Perceptron`

Se implementa la neurona artificial **desde cero** con `numpy`. **Está prohibido** el uso de `scikit-learn`, `tensorflow`, `pytorch` u otros frameworks de IA.

La clase `Perceptron` consta de cuatro métodos:

| Método | Responsabilidad |
|---|---|
| `__init__` | Inicializa la tasa de aprendizaje $\eta$, el número de épocas, el vector de pesos (en ceros) y el sesgo. |
| `activation_function` | Función escalón de **Heaviside**: $1$ si $z \geq 0$, $0$ en caso contrario. |
| `predict` | Calcula $z = \mathbf{w}^T \mathbf{x} + b$ y aplica la función de activación. |
| `fit` | Entrena actualizando pesos y sesgo con la regla de aprendizaje; registra el error por época y aplica **parada temprana** cuando el error llega a cero. |

In [ ]:
# ============================================================================
# PARTE 1: IMPLEMENTACIÓN DE LA CLASE Perceptron DESDE CERO
# ============================================================================

class Perceptron:
    """
    Neurona artificial de un solo nivel: Perceptrón de Rosenblatt.

    Parámetros
    ----------
    input_size : int
        Número de entradas (dimensión del vector x).
    learning_rate : float
        Tasa de aprendizaje eta (0 < eta <= 1).
    epochs : int
        Número máximo de épocas de entrenamiento.
    """

    def __init__(self, input_size, learning_rate=0.1, epochs=100):
        self.lr = learning_rate                 # eta: tasa de aprendizaje
        self.epochs = epochs                    # máximo de épocas permitidas
        self.weights = np.zeros(input_size)     # vector de pesos w (inicializado en ceros)
        self.bias = 0.0                         # sesgo b (bias)
        self.errors_history = []                # errores por época (para la gráfica de convergencia)
        self.convergence_epoch = None           # época en la que convergió (None si no convergió)

    def activation_function(self, z):
        """
        Función de activación: escalón de Heaviside.
        Devuelve 1 si z >= 0, 0 en caso contrario.
        """
        return 1.0 if z >= 0 else 0.0

    def predict(self, x):
        """
        Predice la clase de una muestra x.

        z = w^T x + b  ->  f(z)
        """
        z = np.dot(x, self.weights) + self.bias
        return self.activation_function(z)

    def fit(self, X, y):
        """
        Entrena el Perceptrón aplicando la regla de aprendizaje de Rosenblatt:

            w_j  <-  w_j + eta * (y - y_hat) * x_j
            b    <-  b   + eta * (y - y_hat)

        X : matriz de entrenamiento (cada fila es una muestra).
        y : vector de etiquetas reales (0 o 1).

        Aplica parada temprana si el error total de una época es cero.
        """
        for epoch in range(1, self.epochs + 1):
            total_errors = 0  # contador de errores dentro de la época actual

            for xi, target in zip(X, y):
                prediction = self.predict(xi)   # y_hat: predicción del modelo
                error = target - prediction     # e = y - y_hat

                # Si hay error (e != 0) se aplica la regla de actualización
                if error != 0:
                    self.weights += self.lr * error * np.asarray(xi)
                    self.bias += self.lr * error
                    total_errors += 1

            self.errors_history.append(total_errors)

            # Criterio de parada temprana: error cero -> convergencia
            if total_errors == 0:
                self.convergence_epoch = epoch
                print(f"Convergencia alcanzada en la Época {epoch}")
                break
        else:
            # El bucle terminó sin alcanzar error cero (no hubo break)
            self.convergence_epoch = None
            print(f"No convergió tras {self.epochs} épocas "
                  f"(errores en la última época: {self.errors_history[-1]}).")

# PARTE 2 y 3: Simulación y Visualización de Experimentos

Se definen los **conjuntos de entrenamiento** de las tres compuertas lógicas. Todas comparten la misma matriz de entradas:

$$
X = \begin{bmatrix} 0 & 0 \\ 0 & 1 \\ 1 & 0 \\ 1 & 1 \end{bmatrix}, \qquad
y_{AND} = \begin{bmatrix} 0 \\ 0 \\ 0 \\ 1 \end{bmatrix}, \qquad
y_{OR} = \begin{bmatrix} 0 \\ 1 \\ 1 \\ 1 \end{bmatrix}, \qquad
y_{XOR} = \begin{bmatrix} 0 \\ 1 \\ 1 \\ 0 \end{bmatrix}
$$

Además se definen dos funciones auxiliares de visualización (con `matplotlib`) que se reutilizarán en los tres experimentos:

1. **`plot_convergence`**: gráfica *Épocas vs. Número de errores*.
2. **`plot_decision_boundary`**: gráfica del *límite de decisión* en 2D, que traza la recta

$$
x_2 = -\frac{w_1 x_1 + b}{w_2}
$$

obtenida al despejar $x_2$ de la ecuación $w_1 x_1 + w_2 x_2 + b = 0$, y que sombrea la región donde el modelo predice la clase $1$.

In [ ]:
# ============================================================================
# PARTE 2: DEFINICIÓN DE LOS DATASETS (X, y)
# ============================================================================

# Matriz de entradas (idéntica para las tres compuertas)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

# Vectores de etiquetas
y_and = np.array([0, 0, 0, 1])   # Compuerta AND
y_or  = np.array([0, 1, 1, 1])   # Compuerta OR
y_xor = np.array([0, 1, 1, 0])   # Compuerta XOR

print("X (entradas):\n", X)
print("y_AND:", y_and)
print("y_OR :", y_or)
print("y_XOR:", y_xor)


# ============================================================================
# PARTE 3: FUNCIONES AUXILIARES DE VISUALIZACIÓN (matplotlib)
# ============================================================================

def plot_convergence(perceptron, title):
    """
    Gráfica de convergencia: Épocas vs. Número de errores.
    Muestra cómo evoluciona el error a lo largo del entrenamiento.
    """
    epochs = np.arange(1, len(perceptron.errors_history) + 1)
    errors = np.array(perceptron.errors_history)

    plt.figure(figsize=(6, 4))
    plt.plot(epochs, errors, marker="o", color="#1f77b4", linewidth=2,
             label="Errores por época")
    plt.title(title)
    plt.xlabel("Épocas")
    plt.ylabel("Número de errores")
    plt.xticks(epochs)
    plt.ylim(-0.2, None)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()


def plot_decision_boundary(perceptron, X, y, title):
    """
    Límite de decisión (decision boundary) en 2D.

    - Puntos de las clases 0 (azul) y 1 (rojo) diferenciados por color.
    - Recta divisoria despejando x2 de  w1*x1 + w2*x2 + b = 0:
          x2 = -(w1*x1 + b) / w2
    - Región sombreada: zona donde el modelo predice la clase 1.
    """
    w1, w2 = perceptron.weights
    b = perceptron.bias

    # Rejilla fina para pintar la región de decisión
    x_lin = np.linspace(-0.4, 1.4, 150)
    X1, X2 = np.meshgrid(x_lin, x_lin)
    Z = np.array([perceptron.predict(np.array([xx, yy]))
                  for xx, yy in zip(X1.ravel(), X2.ravel())]).reshape(X1.shape)
    plt.contourf(X1, X2, Z, levels=[-0.5, 0.5],
                 colors=["#cfe8ff", "#ffe0c0"], alpha=0.6)

    # Datos de entrenamiento: clase 1 en rojo, clase 0 en azul
    for xi, yi in zip(X, y):
        color = "#d62728" if yi == 1 else "#1f77b4"
        plt.scatter(xi[0], xi[1], c=color, s=180,
                    edgecolors="k", linewidths=1.2, zorder=3)

    # Recta de decisión:  x2 = -(w1*x1 + b) / w2
    if abs(w2) > 1e-12:
        xx = np.linspace(-0.4, 1.4, 200)
        yy = -(w1 * xx + b) / w2
        plt.plot(xx, yy, color="green", linewidth=2.2,
                 label=f"Frontera: $x_2 = -({w1:.2f}\\,x_1 {b:+.2f})\\,/\\,{w2:.2f}$")
    else:
        # Caso degenerado w2 = 0 (frontera vertical): x1 = -b / w1
        x1b = -b / w1 if abs(w1) > 1e-12 else 0.0
        plt.axvline(x1b, color="green", linewidth=2.2,
                    label=f"Frontera: $x_1 = {x1b:.2f}$")

    plt.title(title)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.xlim(-0.4, 1.4)
    plt.ylim(-0.4, 1.4)
    plt.legend(loc="best")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.show()

## Experimento 1: Compuerta lógica AND

Tabla de verdad de la AND:

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

Entrenamos el Perceptrón y generamos las gráficas de **convergencia** y de **límite de decisión**.

In [ ]:
# ============================================================================
# EXPERIMENTO 1: ENTRENAMIENTO DEL PERCEPTRÓN CON LA COMPUERTA AND
# ============================================================================
p_and = Perceptron(input_size=2, learning_rate=0.1, epochs=100)
p_and.fit(X, y_and)

print(f"Pesos finales: w1 = {p_and.weights[0]:.4f}, "
      f"w2 = {p_and.weights[1]:.4f}, b = {p_and.bias:.4f}")
print(f"Época de convergencia: {p_and.convergence_epoch}")

In [ ]:
# Gráfica de convergencia: Épocas vs. Número de errores (AND)
plot_convergence(p_and, "Convergencia del Perceptrón - Compuerta AND")

In [ ]:
# Gráfica de límite de decisión (AND)
plot_decision_boundary(p_and, X, y_and, "Límite de decisión - Compuerta AND")

## Experimento 2: Compuerta lógica OR

Tabla de verdad de la OR:

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

Entrenamos el Perceptrón y generamos las gráficas de **convergencia** y de **límite de decisión**.

In [ ]:
# ============================================================================
# EXPERIMENTO 2: ENTRENAMIENTO DEL PERCEPTRÓN CON LA COMPUERTA OR
# ============================================================================
p_or = Perceptron(input_size=2, learning_rate=0.1, epochs=100)
p_or.fit(X, y_or)

print(f"Pesos finales: w1 = {p_or.weights[0]:.4f}, "
      f"w2 = {p_or.weights[1]:.4f}, b = {p_or.bias:.4f}")
print(f"Época de convergencia: {p_or.convergence_epoch}")

In [ ]:
# Gráfica de convergencia: Épocas vs. Número de errores (OR)
plot_convergence(p_or, "Convergencia del Perceptrón - Compuerta OR")

In [ ]:
# Gráfica de límite de decisión (OR)
plot_decision_boundary(p_or, X, y_or, "Límite de decisión - Compuerta OR")

## Experimento 3: El reto histórico de Minsky & Papert (Compuerta XOR)

La **XOR** (disyunción exclusiva) devuelve $1$ únicamente cuando las entradas son distintas:

| $x_1$ | $x_2$ | $y$ |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Este problema fue señalado por **Minsky y Papert (1969, *Perceptrons*)** para demostrar la limitación fundamental del Perceptrón simple: *el patrón no es linealmente separable*. De todos modos entrenamos el modelo para observar, en las gráficas, la causa geométrica del fallo.

In [ ]:
# ============================================================================
# EXPERIMENTO 3: ENTRENAMIENTO DEL PERCEPTRÓN CON LA COMPUERTA XOR
# ============================================================================
p_xor = Perceptron(input_size=2, learning_rate=0.1, epochs=100)
p_xor.fit(X, y_xor)

print(f"Pesos finales: w1 = {p_xor.weights[0]:.4f}, "
      f"w2 = {p_xor.weights[1]:.4f}, b = {p_xor.bias:.4f}")
print(f"Época de convergencia: {p_xor.convergence_epoch}")

In [ ]:
# Gráfica de convergencia: Épocas vs. Número de errores (XOR)
plot_convergence(p_xor, "Convergencia del Perceptrón - Compuerta XOR (no converge)")

In [ ]:
# Gráfica de límite de decisión (XOR): se grafica el resultado final aunque no converja
plot_decision_boundary(p_xor, X, y_xor, "Límite de decisión - Compuerta XOR (fracaso esperado)")

## Análisis de convergencia: variación de la tasa de aprendizaje ($\eta$)

Se repite el entrenamiento de las compuertas **AND** y **OR** con varias tasas de aprendizaje ($\eta = 0.01$, $\eta = 0.1$ y $\eta = 0.5$) para estudiar cómo afecta la velocidad de convergencia.

In [ ]:
# ============================================================================
# VARIACIÓN DE LA TASA DE APRENDIZAJE (η)
# ============================================================================
eta_values = [0.01, 0.1, 0.5]
gates = {"AND": y_and, "OR": y_or}

convergence_table = {}
for name, y in gates.items():
    for eta in eta_values:
        model = Perceptron(input_size=2, learning_rate=eta, epochs=1000)
        model.fit(X, y)
        convergence_table[(name, eta)] = model.convergence_epoch

# Tabla resumen de épocas de convergencia
print(f"{'Compuerta':<10} {'η = 0.01':<10} {'η = 0.1':<10} {'η = 0.5':<10}")
print("-" * 40)
for name in gates:
    row = [convergence_table[(name, eta)] for eta in eta_values]
    print(f"{name:<10} {row[0]:<10} {row[1]:<10} {row[2]:<10}")

# Gráfica comparativa: épocas para converger vs. η
fig, ax = plt.subplots(figsize=(7, 4))
bar_width = 0.25
positions = np.arange(len(eta_values))
for i, (name, y) in enumerate(gates.items()):
    values = [convergence_table[(name, eta)] for eta in eta_values]
    ax.bar(positions + i * bar_width, values, bar_width,
           label=name, edgecolor="k", alpha=0.85)
ax.set_xticks(positions + bar_width / 2)
ax.set_xticklabels([f"η = {eta}" for eta in eta_values])
ax.set_xlabel("Tasa de aprendizaje (η)")
ax.set_ylabel("Épocas para converger")
ax.set_title("Épocas de convergencia vs. tasa de aprendizaje (η)")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()

# PARTE 4: Cuestionario y entregables

Respuestas formales de análisis al cuestionario de la práctica.

## Pregunta 1: Análisis de Convergencia

### ¿En cuántas épocas convergió el Perceptrón para la compuerta AND y para la compuerta OR?

Ejecutando el entrenamiento con $\eta = 0.1$ (valor por defecto) se obtiene:

| Compuerta | Épocas hasta converger | Historial de errores por época |
|---|---|---|
| **AND** | **4** | $[2, 3, 3, 0]$ |
| **OR** | **4** | $[2, 2, 1, 0]$ |

Ambas compuertas son **linealmente separables**; por el **Teorema de Convergencia del Perceptrón** (Rosenblatt), el algoritmo garantiza alcanzar una solución en un número finito de pasos siempre que exista una separación lineal. En ambos casos el **criterio de parada temprana** detuvo el entrenamiento en la **época 4**, cuando el número de errores llegó a cero.

### ¿Cómo afecta variar la tasa de aprendizaje ($\eta = 0.01$ vs. $\eta = 0.5$)?

Épocas de convergencia medidas en la simulación de este cuaderno:

| Compuerta | $\eta = 0.01$ | $\eta = 0.1$ | $\eta = 0.5$ |
|---|---|---|---|
| AND | 6 | 4 | 6 |
| OR | 4 | 4 | 4 |

Interpretación técnica:

- **$\eta$ pequeña ($0.01$)**: cada corrección desplaza los pesos en pasos muy cortos, pues $\Delta w_j = \eta \cdot e \cdot x_j$. El proceso es estable pero lento: la **AND** necesita **6 épocas** (más que las 4 de $\eta = 0.1$).
- **$\eta$ moderada ($0.1$)**: punto de equilibrio; la **AND** converge en el menor número de épocas registrado (**4**).
- **$\eta$ grande ($0.5$)**: los saltos en el espacio de pesos son grandes, el algoritmo puede **rebasar** la frontera óptima y necesita épocas adicionales de corrección (la AND vuelve a requerir **6 épocas**). Aun así **converge**, porque el conjunto sigue siendo linealmente separable.
- En la **OR** el resultado fue invariante (**4 épocas** en todos los casos): las muestras corregidas ya apuntan a la dirección correcta, y $\eta$ solo escala la magnitud de los pesos finales.

**Conclusión técnica**: el número de épocas *no es estrictamente monótono* respecto de $\eta$ (depende del orden de actualización *online* y de la geometría de las muestras); sin embargo, la **magnitud de los pesos finales sí escala con $\eta$** (AND: $\mathbf{w} = [0.2, 0.1]$ con $\eta = 0.1$, vs. $\mathbf{w} = [1.0, 0.5]$ con $\eta = 0.5$). En la práctica conviene una tasa moderada: lo bastante grande para converger rápido, sin que el "overshoot" añada ruido ni épocas de más.

## Pregunta 2: El problema de la Separabilidad Lineal (XOR)

### ¿Qué sucedió al intentar entrenar el Perceptrón con la compuerta XOR?

**El Perceptrón no convergió.** En la gráfica de convergencia se observa que el número de errores por época **nunca llega a cero**: oscila entre 3 y 4 en las primeras épocas y luego queda **estabilizado en 4 errores** hasta agotar las 100 épocas. El modelo no encuentra un vector de pesos que clasifique correctamente las cuatro muestras y el entrenamiento termina **sin** satisfacer el criterio de parada temprana.

### Explicación geométrica a partir de la frontera de decisión

El Perceptrón divide el plano mediante una **única recta** (la frontera de decisión):

$$
w_1 x_1 + w_2 x_2 + b = 0 \quad\Longrightarrow\quad x_2 = -\frac{w_1 x_1 + b}{w_2}
$$

Esa recta separa el plano en dos **semiespacios**: de un lado todos los puntos se clasifican como $0$ y del otro como $1$. En el problema XOR las clases están dispuestas **en diagonal** (patrón de ajedrez en las esquinas del cuadrado $[0,1] \times [0,1]$):

```
    (0, 1) = 1          (1, 1) = 0

    (0, 0) = 0          (1, 0) = 1
```

Los dos unos, $(0,1)$ y $(1,0)$, están **diagonalmente opuestos**, y los dos ceros, $(0,0)$ y $(1,1)$, forman la otra diagonal. **Ninguna línea recta puede separarlos**:

1. Una recta horizontal deja a $(0,1)$ y $(1,1)$ del mismo lado — pero son de **distinta clase**.
2. Una recta vertical deja a $(0,0)$ y $(0,1)$ del mismo lado — también de **distinta clase**.
3. Cualquier recta inclinada divide el cuadrado en **dos bloques contiguos**; la partición que exige la XOR es **alternada** (cada esquina es de distinta clase que sus dos vecinas), lo que requeriría una curva que se "doble", imposible para una línea.

Por ello la gráfica final —aunque el modelo no converja— siempre deja **al menos un punto mal clasificado**; de hecho, la mejor recta posible sobre XOR separa como máximo **3 de los 4** puntos. Este es precisamente el fenómeno que **Minsky y Papert formalizaron en 1969**: el Perceptrón simple solo resuelve problemas **linealmente separables** (AND y OR), y como la XOR no lo es, se requiere una arquitectura multicapa (MLP) con una **capa oculta** y activaciones no lineales.

## Pregunta 3: Interpretación de Pesos (compuerta AND)

### Valores finales obtenidos ($\eta = 0.1$)

Entrenando el Perceptrón con la compuerta AND se alcanzaron los siguientes parámetros:

$$
w_1 = 0.20, \qquad w_2 = 0.10, \qquad b = -0.20
$$

(El sesgo reportado por la máquina es $b = -0.20000000000000004$; es únicamente la representación en **coma flotante** de $-0.2$.)

La neurona implementa la regla de decisión:

$$
z = 0.20\,x_1 + 0.10\,x_2 - 0.20
\qquad\Longrightarrow\qquad
\hat{y} =
\begin{cases}
1 & \text{si } z \geq 0 \\[2mm]
0 & \text{si } z < 0
\end{cases}
$$

### Significado físico de cada número

- **$w_1 = 0.20$ y $w_2 = 0.10$ (pesos sinápticos):** miden la **fuerza de la conexión** de cada entrada con la neurona. Ambos son **positivos**, lo que significa que ambas entradas aportan evidencia *a favor* de la clase $1$: cuanto mayor es la entrada, mayor es $z$. Como $w_1 > w_2$, la entrada $x_1$ influye más en la decisión (en este caso la asimetría es un artefacto del **orden de actualización online**, no una diferencia semántica entre variables).
- **$b = -0.20$ (sesgo / umbral):** desplaza la frontera y actúa como *nivel de disparo*. La neurona emite $1$ solo si la evidencia supera el umbral: $0.20\,x_1 + 0.10\,x_2 \geq 0.20$. Como la suma máxima activable con **una sola** entrada a $1$ es exactamente $0.20$ (en el umbral), la unidad exige que **ambas** entradas estén activas (alcanzando $z = 0.30$ en $(1,1)$): por eso replica la lógica **AND**.
- **Frontera de decisión:**

$$
x_2 = -\frac{w_1 x_1 + b}{w_2} = -\frac{0.20\,x_1 - 0.20}{0.10} = 2 - 2\,x_1
$$

### Verificación sobre la tabla de verdad

| $(x_1, x_2)$ | $z = 0.20x_1 + 0.10x_2 - 0.20$ | $\hat{y}$ | $y$ | ¿Correcto? |
|---|---|---|---|---|
| (0, 0) | $-0.20$ | 0 | 0 | ✓ |
| (0, 1) | $-0.10$ | 0 | 0 | ✓ |
| (1, 0) | $\approx 0$ ($\approx -4.4\times10^{-17}$) | 0 | 0 | ✓ |
| (1, 1) | $+0.10$ | 1 | 1 | ✓ |

**Observación adicional:** la recta $x_2 = 2 - 2x_1$ pasa **exactamente** por el vértice $(1,0)$; la muestra queda del lado correcto solo gracias a que el sesgo es un poco menor que $-0.2$ ($b = -0.20000000000000004$). Esto ilustra que el Perceptrón clásico **no maximiza el margen**: a diferencia de una SVM, se detiene apenas el error es cero, aunque la frontera quede pegada a los datos.

## Conclusiones

1. El Perceptrón simple, implementado desde cero con `numpy`, resuelve correctamente problemas **linealmente separables** (AND y OR) y lo hace en muy pocas épocas (**4 épocas** con $\eta = 0.1$).
2. La tasa de aprendizaje $\eta$ controla la magnitud de las correcciones y, por tanto, la velocidad de convergencia y la magnitud de los pesos finales, aunque su efecto sobre el número de épocas no es estrictamente monótono.
3. El fracaso en XOR evidencia la limitación geométrica del modelo: una única frontera lineal no puede separar clases dispuestas en diagonal; se necesita una arquitectura con capa oculta (MLP).
4. Todo el desarrollo cumple las restricciones de la práctica: `numpy` para la lógica matemática y `matplotlib` únicamente para la visualización, sin `scikit-learn` ni otros frameworks de IA.